In [2]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Visualization & Metrics
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Performance / device setup
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
USE_GPU = len(gpus) > 0
if USE_GPU:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("GPUs:", gpus)
else:
    print("GPUs: none (TensorFlow will use CPU).")
    print("Tip: On Windows, TensorFlow GPU is typically available via WSL2 (Ubuntu) + NVIDIA drivers.")

# XLA (jit_compile) usually helps on GPU
USE_JIT = USE_GPU

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
DATASET_PATH = "Detect_solar_dust"
import os
print("Contents of dataset folder:", os.listdir(DATASET_PATH))

for d in os.listdir(DATASET_PATH):
      folder = os.path.join(DATASET_PATH, d)
      if os.path.isdir(folder):
          print(f"{d} → {len(os.listdir(folder))} files")


Contents of dataset folder: ['Clean', 'Dusty']
Clean → 1493 files
Dusty → 1069 files


In [4]:
import os
import shutil
import tensorflow as tf
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 1337
AUTOTUNE = tf.data.AUTOTUNE

# Force clear cache
CACHE_DIR = os.path.join(os.getcwd(), ".tf_data_cache_solar")
if os.path.exists(CACHE_DIR):
    print(f"Clearing old cache at {CACHE_DIR}...")
    try:
        shutil.rmtree(CACHE_DIR)
    except Exception as e:
        print(f"Warning: Could not delete cache. Delete '{CACHE_DIR}' manually.")
os.makedirs(CACHE_DIR, exist_ok=True)

# MUCH stronger augmentation to slow down memorization
# The dataset is small (~2000 train images), so we need to make each image 
# look very different every epoch to prevent the model from memorizing pixel patterns.
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),  # Both axes
        tf.keras.layers.RandomRotation(factor=0.2, fill_mode="nearest"),  # ±72 degrees
        tf.keras.layers.RandomTranslation(height_factor=0.2, width_factor=0.2, fill_mode="nearest"),
        tf.keras.layers.RandomZoom(height_factor=(-0.3, 0.3), fill_mode="nearest"),
        tf.keras.layers.RandomBrightness(factor=0.2),  # Doubled
        tf.keras.layers.RandomContrast(factor=0.2),  # Doubled
    ],
    name="data_augmentation",
)

train_raw = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_raw = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

# Verify class balance
print("\nVerifying Validation Split Balance...")
y_val = []
for _, labels in val_raw:
    y_val.extend(labels.numpy().flatten())
unique, counts = np.unique(y_val, return_counts=True)
print(f"Validation Class Distribution: {dict(zip(unique, counts))}")
print(f"Class Names: {train_raw.class_names}\n")

def train_preprocess(images, labels):
    images = data_augmentation(images, training=True)
    images = preprocess_input(tf.cast(images, tf.float32))
    return images, labels

def val_preprocess(images, labels):
    images = preprocess_input(tf.cast(images, tf.float32))
    return images, labels

# NO caching for train - forces fresh augmentation every epoch (critical for small data)
train_data = (
    train_raw
    .map(train_preprocess, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_data = (
    val_raw
    .map(val_preprocess, num_parallel_calls=AUTOTUNE)
    .cache(os.path.join(CACHE_DIR, "val"))
    .prefetch(AUTOTUNE)
)

Clearing old cache at /home/sanjeev/solar_panel_project/.tf_data_cache_solar...
Found 2562 files belonging to 2 classes.
Using 2050 files for training.


I0000 00:00:1772971128.430202    5616 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 2562 files belonging to 2 classes.
Using 512 files for validation.

Verifying Validation Split Balance...


2026-03-08 11:58:50.396563: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: known incorrect sRGB profile
2026-03-08 11:58:50.396621: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: cHRM chunk does not match sRGB


Validation Class Distribution: {np.float32(0.0): np.int64(290), np.float32(1.0): np.int64(222)}
Class Names: ['Clean', 'Dusty']



In [5]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
base_model.trainable = False

In [6]:
# Classification Head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

output = Dense(1, activation="sigmoid", dtype="float32")(x)

model = Model(inputs=base_model.input, outputs=output)

# Lower initial LR to slow down learning on head (prevents early memorization)
model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
    jit_compile=USE_JIT,
)

In [7]:
# Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("best_resnet_solar.keras", monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

# Phase 1: Train just the head (more epochs since augmentation is heavier now)
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25


2026-03-08 11:58:55.440202: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
2026-03-08 11:59:06.776872: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
I0000 00:00:1772971163.772845    7277 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


16/65 ━━━━━━━━━━━━━━━━━━━━ 5s 120ms/step - accuracy: 0.5386 - loss: 0.7751

2026-03-08 11:59:25.619580: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: known incorrect sRGB profile
2026-03-08 11:59:25.619617: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: cHRM chunk does not match sRGB
Corrupt JPEG data: 1 extraneous bytes before marker 0xed


24/65 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - accuracy: 0.5472 - loss: 0.7615

...

30/65 ━━━━━━━━━━━━━━━━━━━━ 8s 244ms/step - accuracy: 0.7693 - loss: 0.4819

Corrupt JPEG data: 1 extraneous bytes before marker 0xed
2026-03-08 12:11:26.341526: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: known incorrect sRGB profile
2026-03-08 12:11:26.342019: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: cHRM chunk does not match sRGB


47/65 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - accuracy: 0.7696 - loss: 0.4803

2026-03-08 12:11:43.071608: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: known incorrect sRGB profile
2026-03-08 12:11:43.071656: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: cHRM chunk does not match sRGB


64/65 ━━━━━━━━━━━━━━━━━━━━ 0s 621ms/step - accuracy: 0.7699 - loss: 0.4799
Epoch 25: val_accuracy did not improve from 0.81250
65/65 ━━━━━━━━━━━━━━━━━━━━ 42s 626ms/step - accuracy: 0.7751 - loss: 0.4766 - val_accuracy: 0.8008 - val_loss: 0.4762 - learning_rate: 5.0000e-05
Restoring model weights from the end of the best epoch: 23.


In [ ]:
# Phase 2: Fine-Tuning
# Keep BatchNorm frozen. Only unfreeze conv layers in last 50.

base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-50:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=5e-6),  # Very low - just nudge, don't retrain
    loss="binary_crossentropy",
    metrics=["accuracy"],
    jit_compile=USE_JIT,
)

print("Trainable variables:", len(model.trainable_variables))

fine_tune_history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=callbacks
)

Trainable variables: 16
Epoch 1/25


2026-03-08 12:12:21.032687: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
2026-03-08 12:12:21.442563: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 67109120 bytes after encountering the first element of size 67109120 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
2026-03-08 12:12:22.798956: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 34330240 bytes after encountering the first element of size 34330240 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
Corrupt JPEG data: 1 extraneous bytes before marker 0xed


26/65 ━━━━━━━━━━━━━━━━━━━━ 12s 328ms/step - accuracy: 0.7804 - loss: 0.4791

2026-03-08 12:12:52.170565: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: known incorrect sRGB profile
2026-03-08 12:12:52.170632: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: cHRM chunk does not match sRGB


30/65 ━━━━━━━━━━━━━━━━━━━━ 14s 416ms/step - accuracy: 0.7785 - loss: 0.4801

...

2026-03-08 12:15:54.983328: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: known incorrect sRGB profile
2026-03-08 12:15:54.987232: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: cHRM chunk does not match sRGB


48/65 ━━━━━━━━━━━━━━━━━━━━ 10s 605ms/step - accuracy: 0.8105 - loss: 0.4319

2026-03-08 12:16:06.070899: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: known incorrect sRGB profile
2026-03-08 12:16:06.071523: W tensorflow/core/lib/png/png_io.cc:95] PNG warning: iCCP: cHRM chunk does not match sRGB


65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 677ms/step - accuracy: 0.8076 - loss: 0.4336
Epoch 5: val_accuracy did not improve from 0.81445
65/65 ━━━━━━━━━━━━━━━━━━━━ 47s 696ms/step - accuracy: 0.8020 - loss: 0.4357 - val_accuracy: 0.8086 - val_loss: 0.4455 - learning_rate: 5.0000e-06
Epoch 6/25


2026-03-08 12:16:22.486213: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 67109120 bytes after encountering the first element of size 67109120 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


In [ ]:
val_loss, val_acc = model.evaluate(val_data)
print(f"Validation Accuracy: {val_acc:.4f}")


In [ ]:
# --- VISUALIZATION: Prediction Examples ---
def show_predictions(dataset, model, class_names, num_images=16):
    plt.figure(figsize=(15, 15))
    
    # Take one batch
    for images, labels in dataset.take(1):
        # Predict batch
        preds = model.predict(images, verbose=0)
        
        for i in range(num_images):
            ax = plt.subplot(4, 4, i + 1)
            
            # Image needs to be un-normalized for display if we used preprocess_input
            # ResNet50 preprocess maps to [-1, 1], so (x+1)/2 * 255 puts it back roughly to [0,255]
            img_disp = images[i].numpy()
            img_disp = (img_disp + 1.0) / 2.0 
            plt.imshow(np.clip(img_disp, 0, 1))
            
            # FIX: labels[i] is shape (1,), need to access index 0 to get the scalar
            true_label = class_names[int(labels[i].numpy()[0])]
            
            pred_conf = preds[i][0]
            pred_label = class_names[int(pred_conf > 0.5)]
            
            # Color code title: Green = Correct, Red = Wrong
            color = 'green' if true_label == pred_label else 'red'
            
            plt.title(f"True: {true_label}\nPred: {pred_label} ({pred_conf:.2f})", color=color)
            plt.axis("off")
    plt.show()

# Show 'em
show_predictions(val_data, model, train_raw.class_names)

In [ ]:
# --- VISUALIZATION: Confusion Matrix ---

# 1. Get Predictions
print("Generating predictions for Confusion Matrix...")
y_pred_probs = model.predict(val_data, verbose=1)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

# 2. Get True Labels (Iterate safely to ensure alignment)
y_true = []
for images, labels in val_data:
    y_true.extend(labels.numpy().astype("int32"))
y_true = np.array(y_true)

# 3. Create Matrix
cm = confusion_matrix(y_true, y_pred)
class_names = train_raw.class_names # ['Clean', 'Dusty'] typically

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()

# 4. Text Report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# --- VISUALIZATION: Training Metrics ---
def plot_history(initial_hist, fine_tune_hist=None, initial_epochs=10):
    acc = initial_hist.history['accuracy']
    val_acc = initial_hist.history['val_accuracy']
    loss = initial_hist.history['loss']
    val_loss = initial_hist.history['val_loss']

    if fine_tune_hist:
        acc += fine_tune_hist.history['accuracy']
        val_acc += fine_tune_hist.history['val_accuracy']
        loss += fine_tune_hist.history['loss']
        val_loss += fine_tune_hist.history['val_loss']

    plt.figure(figsize=(12, 6))
    
    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    if fine_tune_hist:
        plt.plot([initial_epochs-1, initial_epochs-1], 
                 plt.ylim(), label='Start Fine Tuning')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    if fine_tune_hist:
        plt.plot([initial_epochs-1, initial_epochs-1], 
                 plt.ylim(), label='Start Fine Tuning')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    
    plt.show()

# Detect if we have fine_tune_history, otherwise just pass None
ft_hist = fine_tune_history if 'fine_tune_history' in locals() else None
init_epochs = len(history.history['accuracy'])
plot_history(history, ft_hist, initial_epochs=init_epochs)